# Understanding Corruption in Europe

## Introduction

Corruption remains a pressing issue across Europe, with significant implications for governance, economic stability, and social equality. However, quantifying corruption, understanding its forms, and identifying predictors of its prevalence is a complex challenge. 

In this notebook, we try to analyze and quantify corruption levels and perceptions in European countries, investigating both actual and perceived corruption. 


## Table of Contents

1. [Question proposal and Hypothesis Definition](#1-questions-and-hypothesis)
   - Problem Framing
   - Hypothesis definition

2. [Get the Data](#2-get-the-data)
   - Selection of datasets
   - Justification of the chosen datasets

3. [Explore the Data](#3-explore-the-data)
   - Data cleaning and transformation
   - Addressing missing values
   - Exploratory Data Analysis (EDA)

4. [Model the Data](#4-model-the-data)

5. [Communicate and Visualize Results](#5-communicate-and-visualize-results)
   - Key findings and insights
   - Potential biases and limitations

6. [Reflection and Learning](#6-reflection-and-learning)
   - Challenges faced
   - Key insights and potential biases
   - Techniques and tools learned
   - Division of work among group members

---

In [20]:
# Imports
# General purpose Python packages
import os
from typing import Union

# Data munging
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import NamedStyle
from openpyxl.utils.dataframe import dataframe_to_rows

# Data viz
from matplotlib import pyplot as plt
import seaborn as sns

# Data modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder

# To ignore warinings
import warnings

warnings.filterwarnings("ignore")

# Env variables
SEED = 42

## 1. Question Proposal and Hypothesis Definition

### Framing the Problem
**Context and Significance of the Research**  
Corruption remains a critical challenge across Europe, impacting governance, social equity, and public trust in institutions. While some countries experience high levels of visible corruption, others may have issues related to institutions or cultural norms. Understanding the relationship between perceived and actual corruption is important for implementing anti-corruption strategies and rebuilding public trust. 

**Initial Exploration of Hypotheses**  
The analysis focuses on the following key questions:
1. What are the differences between perceived and actual corruption across European countries?
2. Are different forms of corruption (e.g., political, administrative) prevalent in specific regions?
3. How do public reporting and attitudes toward corruption influence corruption dynamics?

---

### Defining Hypotheses
1. **Hypothesis 1: Misalignment Between Perceived and Actual Corruption**  
   There is a measurable gap between perceived corruption (e.g., QB5: Perceived Widespread Corruption) and actual corruption (e.g., Tolerance Index to Corruption, CPI_avg). This gap varies significantly by country and sector.

2. **Hypothesis 2: Tolerance and Perceptions of Corruption**  
   Countries with higher tolerance toward corruption (e.g., QB4.1-QB4.3: Acceptability of Corruption) are more likely to experience higher levels of actual corruption (e.g., CPI_percent_change, QB12: Experienced Corruption).

3. **Hypothesis 3: Reporting Behavior and Corruption Levels**  
   Higher public awareness of corruption reporting mechanisms (e.g., QB10: Knows Where to Report Corruption) correlates with lower levels of actual corruption.



## 2. Get the Data

### Selection of Datasets
**Description of Selected Datasets**  
The primary dataset used for this analysis is derived from the **Eurobarometer survey**, a large-scale public opinion survey conducted across European countries. The dataset includes responses to questions regarding both **perceived corruption** (e.g., public attitudes and experiences) and **actual corruption** (e.g., reported incidents, tolerance indices). Additionally, it integrates historical CPI (Corruption Perception Index) data for contextual analysis.

**Reasons for Dataset Choice**  
1. **Comprehensive Coverage:**  
   The Eurobarometer dataset provides a wide range of metrics capturing both subjective perceptions and objective indicators of corruption across multiple European countries.
   
2. **Sectoral Insights:**  
   Specific questions (e.g., QB1-QB3) focus on corruption in sectors such as healthcare and public services, enabling granular analysis of sectoral vulnerabilities.

3. **Temporal Analysis:**  
   Historical CPI data (e.g., CPI 2019-2023) allows for a longitudinal understanding of corruption trends.

4. **Cross-Country Comparisons:**  
   The dataset supports cross-country comparisons, making it possible to cluster countries with similar corruption patterns and identify regional disparities.


### Data Sources
**Sources of Datasets and Their Reliability**  
1. **Eurobarometer Survey**  
   - **Source:** European Commission, public opinion surveys.  
   - **Reliability:** Eurobarometer is a well-established and reputable data source, regularly used for policy-making and academic research. Its standardized methodology ensures high data quality and comparability across countries.

2. **Corruption Perception Index (CPI)**  
   - **Source:** Transparency International, annual corruption perception rankings.  
   - **Reliability:** The CPI is a widely accepted and validated measure of corruption, aggregating expert assessments and opinion surveys from various sources.  
   - **Historical Data:** Includes CPI data from 2019 to 2023, capturing trends over time.


## 3. Data preprocessing

### Data Cleaning and Transformation
- **Handling inconsistencies:**  
  All columns were converted to numeric where applicable, and categorical values were standardized. For example, country codes were cleaned and encoded using label encoding. Outliers in numerical columns were identified and capped using the IQR method to maintain data consistency.


### Addressing Missing Values
- **Methods used:**  
  - Numerical columns were filled using the mean strategy.  
  - Categorical columns were filled with the mode or "Unknown" as a placeholder.


#### CPI Preprocessing

In [7]:
def read_excel_cpi(filename: str, sheet_name: Union[str, int]):
    path = os.path.join("..", "input", "cpi-2019-2023", filename)
    return pd.read_excel(path, sheet_name=sheet_name)


def clean_rows(data: pd.DataFrame, from_row: int = 2):
    # CPI data seems to always have three useless rows at the top of the excel sheet.
    data = data.iloc[from_row:]
    # Make the first row the column names
    data.columns = data.iloc[0]
    # Drop the first row
    data = data.iloc[1:]
    # Reset the index
    data.reset_index(drop=True, inplace=True)
    return data


def enforce_type(data: pd.DataFrame):
    # Enforce all columns except the first 3 to be numeric
    for col in data.columns[3:]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    return data


def load_cpi(filename: str, sheetname: str, from_row: int = 1):
    return enforce_type(clean_rows(read_excel_cpi(filename, sheetname), from_row))


def load_all_cpi():
    return {
        2019: load_cpi("CPI2019-1.xlsx", "CPI2019"),
        2020: load_cpi("CPI2020_GlobalTablesTS_210125.xlsx", "CPI2020"),
        2021: load_cpi("CPI2021_GlobalResults_Trends.xlsx", "CPI 2021"),
        2022: load_cpi("CPI2022_GlobalResultsTrends.xlsx", "CPI 2022 (final)"),
        2023: load_cpi("CPI2023_Global_Results__Trends.xlsx", "CPI 2023", 2),
    }


def concat_cpis(dfs: dict):
    for _, df in dfs.items():
        for col in df.columns:
            if col.startswith("CPI score"):
                df.rename(columns={col: "CPI score"}, inplace=True)
            elif col.startswith("Country /"):
                df.rename(columns={col: "Country"}, inplace=True)
            elif col.startswith("standard error"):
                df.rename(columns={col: "Standard error"}, inplace=True)
            elif col.startswith("Global Insight"):
                df.rename(
                    columns={col: "Global Insights Country Risk Ratings"}, inplace=True
                )
            elif col.startswith("Freedom House Nations in Transit Ratings"):
                df.rename(
                    columns={col: "Freedom House Nations in Transit"}, inplace=True
                )

    # Add a new column for the year and place it between "Region" and "CPI Score"
    for year, df in dfs.items():
        df.insert(3, "Year", year)

    return pd.concat(dfs.values(), ignore_index=True)

#### Corruption Risk preprocessing

In [8]:
def read_excel_corruption_risk(filename: str, sheet_name: Union[str, int]):
    path = os.path.join("..", "input", "corruptionrisk", filename)
    return pd.read_excel(path, sheet_name=sheet_name)


def renaming(df: pd.DataFrame):
    for colname in df.columns:
        col = colname.lower().replace(" ", "")
        if col.startswith("countryname"):
            df.rename(columns={colname: "Country"}, inplace=True)
        elif col.startswith("countrycode"):
            df.rename(columns={colname: "ISO3"}, inplace=True)
        elif col.startswith("ipi"):
            df.rename(columns={colname: "Public Integrity"}, inplace=True)
        elif col.startswith("region"):
            df.rename(columns={colname: "Region"}, inplace=True)


def cleaning_transparency(df: pd.DataFrame):
    # Drop the third column
    df.drop(columns=["Countrycode 2"], inplace=True)
    renaming(df)
    return df


def cleaning_public_integrity(df: pd.DataFrame, year: int):
    # Drop the first 9 rows
    df = df.iloc[9:]
    # Make the first row the header
    df.columns = df.iloc[0]
    # Drop the first row
    df = df.iloc[1:]
    # Add the year column
    df["Year"] = year
    renaming(df)

    if year == 2023:
        # From column 5 onwards, replace all "," by "."
        for col in df.columns[5:]:
            df[col] = df[col].apply(lambda x: str(x).replace(",", "."))

        # Divide the population column by 1e6
        df["population"] = df["population"].apply(lambda x: float(x) / 1e6)

        # Format urban population column:
        # drop last % sign and make it a float
        df["urban population"] = df["urban population"].apply(
            lambda x: float(str(x).replace("%", ""))
        )

    df = enforce_types(df)

    return df


def enforce_types(df: pd.DataFrame):
    # From column 5 onwards, enforce numeric type
    for col in df.columns[5:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def missing_values_integrity(df: pd.DataFrame):
    # NOTE: Starting from the 2021 edition, administrative burden and trade
    # openness have been replaced by administrative transparency and online
    # services, due to unavailable alternative data in the original components
    # (based on the World Bank Doing Business project, which closed).
    # Source: https://corruptionrisk.org/ipi-methodology/

    # This means that missing values in these columns cannot be made up. Bad luck I guess

    # Mean years of schooling has missing data for 2023
    # Impute by extrapolation
    df.set_index(["ISO3", "Year"], inplace=True)

    # Extrapolate data using 2019 and 2022
    for iso3 in df.index.levels[0]:
        try:
            if pd.isna(df.loc[(iso3, 2023), "mean years of schooling"]):
                value_2019 = df.loc[(iso3, 2019), "mean years of schooling"]
                value_2022 = df.loc[(iso3, 2021), "mean years of schooling"]
                extrapolated_value = value_2022 + (value_2022 - value_2019) / 2
                df.loc[(iso3, 2023), "mean years of schooling"] = extrapolated_value
        except KeyError:
            # Not enough data to interpolate, these missing values will remain
            pass

    # Venezuela has data for 2019 regarding income class, but not for 2021 and 2023.
    # Simply copy the 2019 data to 2021 and 2023
    for year in [2021, 2023]:
        if pd.isna(df.loc[("VEN", year), "income_class "]):
            df.loc[("VEN", year), "income_class "] = df.loc[
                ("VEN", 2019), "income_class "
            ]
        if pd.isna(df.loc[("VEN", year), "income_class_shortcode"]):
            df.loc[("VEN", year), "income_class_shortcode"] = df.loc[
                ("VEN", 2019), "income_class_shortcode"
            ]

    # North Macedonia and DR Congo have missing values for urban population in
    # 2021 and 2023 respectively
    # Impute by linear interpolation/extrapolation
    df.loc[("MKD", 2021), "urban population"] = (
        df.loc[("MKD", 2019), "urban population"]
        + df.loc[("MKD", 2023), "urban population"]
    ) / 2

    df.loc[("COD", 2023), "urban population"] = (
        df.loc[("COD", 2021), "urban population"]
        - df.loc[("COD", 2019), "urban population"]
    ) / (2021 - 2019) * (2023 - 2019) + df.loc[("COD", 2019), "urban population"]

    # Despite the efforts, the data for these two countries (and maybe for more)
    # is botched. I can already they are outliers (at least in the population column)

    df.reset_index(inplace=True)
    return df


def concat_public_integrity(df: list):
    # Concatenate the public integrity index
    return pd.concat(df)

#### Perceived and Actual corruption preprocessing 

In [5]:
def handle_missing_values(df, strategy="mean"):
    for col in df.columns:
        if df[col].dtype in ["float64", "int64"]:
            if strategy == "mean":
                df[col] = df[col].fillna(df[col].mean())
            elif strategy == "median":
                df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].replace("- ", np.nan)
            df[col] = df[col].astype(str).fillna(df[col].mode()[0])
    return df


def encode_categorical(df, columns):
    label_encoders = {}
    for col in columns:
        df[col] = df[col].astype(str).str.strip().replace("- ", np.nan)
        df[col] = df[col].fillna("Unknown")
        le = LabelEncoder()
        encoded_col_name = f"{col}_ID"
        df[encoded_col_name] = le.fit_transform(df[col])
        label_encoders[col] = le
    return df, label_encoders


def normalize_features(df, columns):
    scaler = StandardScaler()
    df[columns] = scaler.fit_transform(df[columns])
    return df, scaler


def detect_outliers(df, columns):
    for col in columns:
        if df[col].dtype in ["float64", "int64"]:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    return df


def create_composite_indices(df):
    df["QB4T"] = df[["QB4.1", "QB4.2", "QB4.3"]].mean(axis=1)
    return df


def add_coordinates(df):
    coordinates = {
        "BE": (50.8503, 4.3517),
        "BG": (42.6977, 23.3219),
        "CZ": (50.0755, 14.4378),
        "DK": (55.6761, 12.5683),
        "DE": (51.1657, 10.4515),
        "EE": (59.4370, 24.7536),
        "IE": (53.3498, -6.2603),
        "EL": (37.9838, 23.7275),
        "ES": (40.4168, -3.7038),
        "FR": (48.8566, 2.3522),
        "HR": (45.8150, 15.9819),
        "IT": (41.9028, 12.4964),
        "CY": (35.1264, 33.4299),
        "LV": (56.9496, 24.1052),
        "LT": (54.6872, 25.2797),
        "LU": (49.8153, 6.1296),
        "HU": (47.4979, 19.0402),
        "MT": (35.8997, 14.5146),
        "NL": (52.3676, 4.9041),
        "AT": (48.2082, 16.3738),
        "PL": (52.2297, 21.0122),
        "PT": (38.7169, -9.1390),
        "RO": (44.4268, 26.1025),
        "SI": (46.0569, 14.5058),
        "SK": (48.1486, 17.1077),
        "FI": (60.1695, 24.9354),
        "SE": (59.3293, 18.0686),
        "UK": (51.5074, -0.1278),
    }
    df["Latitude"] = df["Country"].map(lambda x: coordinates.get(x, (None, None))[0])
    df["Longitude"] = df["Country"].map(lambda x: coordinates.get(x, (None, None))[1])
    return df


def calculate_additional_features(df):
    qb_columns = [col for col in df.columns if col.startswith("QB")]
    if qb_columns:
        df["QB_avg"] = df[qb_columns].mean(axis=1)

    if "QB6" in df.columns and "QB13" in df.columns:
        df["QB6_QB13"] = df["QB6"] * df["QB13"]

    if "QB15.6" in df.columns and "QB15.7" in df.columns:
        df["QB15.6_QB15.7"] = df["QB15.6"] * df["QB15.7"]

    cpi_columns = ["CPI 2020", "CPI 2021", "CPI 2022", "CPI 2023"]
    if all(col in df.columns for col in cpi_columns):
        df["CPI_avg"] = df[cpi_columns].mean(axis=1)
        for year in ["2020", "2021", "2022", "2023"]:
            col_name = f"CPI_{year}_minus_avg"
            df[col_name] = df[f"CPI {year}"] - df["CPI_avg"]

        df["CPI_percent_change"] = (df["CPI 2023"] - df["CPI 2020"]) / df["CPI 2020"]

        df["CPI_category"] = pd.cut(
            df["CPI_avg"],
            bins=[0, 50, 70, float("inf")],
            labels=["Low", "Medium", "High"],
            right=False,
        )
        category_mapping = {"Low": 0, "Medium": 1, "High": 2}
        df["CPI_category_encoded"] = df["CPI_category"].map(category_mapping)

    if "df_totalscore" in df.columns:
        df["TotalScore_category"] = pd.cut(
            df["df_totalscore"],
            bins=[-float("inf"), 5, 10, float("inf")],
            labels=["Low", "Medium", "High"],
            right=False,
        )
        score_mapping = {"Low": 0, "Medium": 1, "High": 2}
        df["TotalScore_category_encoded"] = df["TotalScore_category"].map(score_mapping)

    return df


def preprocess_eurobarometer(CORRUPTION_DF: pd.DataFrame) -> pd.DataFrame:
    for col in CORRUPTION_DF.columns:
        if col != "Country":
            CORRUPTION_DF[col] = pd.to_numeric(CORRUPTION_DF[col], errors="coerce")

    CORRUPTION_DF = handle_missing_values(CORRUPTION_DF, strategy="mean")
    numerical_columns = [
        "QB1",
        "QB2",
        "QB10",
        "QB15.1",
        "QB15.2",
        "QB15.3",
        "CPI 2020",
        "CPI 2021",
        "CPI 2022",
        "CPI 2023",
    ]
    CORRUPTION_DF = detect_outliers(CORRUPTION_DF, numerical_columns)
    CORRUPTION_DF = add_coordinates(CORRUPTION_DF)

    categorical_columns = ["Country"]
    CORRUPTION_DF, _ = encode_categorical(CORRUPTION_DF, categorical_columns)

    CORRUPTION_DF = create_composite_indices(CORRUPTION_DF)
    CORRUPTION_DF = calculate_additional_features(CORRUPTION_DF)

    columns_order = [
        "Country",
        "Country_ID",
        "QB1",
        "QB10",
        "QB15.1",
        "QB15.10",
        "QB15.11",
        "QB15.12",
        "QB15.13",
        "QB15.2",
        "QB15.3",
        "QB15.4",
        "QB15.5",
        "QB15.6",
        "QB15.7",
        "QB15.8",
        "QB15.9",
        "QB4.1",
        "QB4.2",
        "QB4.3",
        "QB4T",
        "QB5",
        "QB6",
        "QB12",
        "QB13",
        "QB2",
        "QB6_QB13",
        "QB15.6_QB15.7",
        "QB_avg",
        "CPI 2020",
        "CPI 2021",
        "CPI 2022",
        "CPI 2023",
        "CPI_avg",
        "CPI_2020_minus_avg",
        "CPI_2021_minus_avg",
        "CPI_2022_minus_avg",
        "CPI_2023_minus_avg",
        "CPI_percent_change",
        "CPI_category",
        "CPI_category_encoded",
        "TotalScore_category",
        "TotalScore_category_encoded",
        "df_totalscore",
        "df_fulfillment",
        "dj_total score",
        "dj_fulfillment",
        "tindex_total",
        "tindex_fulfillment",
        "Latitude",
        "Longitude",
    ]

    CORRUPTION_DF = CORRUPTION_DF[
        [col for col in columns_order if col in CORRUPTION_DF.columns]
    ]

    return CORRUPTION_DF


def load_eurobarometer() -> pd.DataFrame:
    CORRUPTION_DATASET_NAME = "euro-barometer-ebs-a-v1230"
    CORRUPTION_FILE_NAME = "Corruption_ebs_A_v1230.xlsx"
    CORRUPTION_PATH = f"/kaggle/input/{CORRUPTION_DATASET_NAME}"
    CORRUPTION_FILE_PATH = os.path.join(CORRUPTION_PATH, CORRUPTION_FILE_NAME)
    df = pd.read_excel(os.path.join(CORRUPTION_FILE_PATH), sheet_name=1)
    return preprocess_eurobarometer(df)

#### Eurostat
##### Recorded crimes

In [18]:
def preprocess_sheet(sheet_name, df):
    """
    Clean the sheets and bring them in an long format
    """
    try:
        # Clean empty rows
        df = df.dropna(how="all").reset_index(drop=True)

        # Setting the header
        columns = (
            ["GEO"]
            + [f"Year_{i}" for i in range(2013, 2023)]
            + [f"Unused_{i}" for i in range(2013, 2023)]
        )
        df.columns = columns[: len(df.columns)]

        # Drop irrelevant rows
        df = df.dropna(subset=["GEO"])
        df = df[
            ~df["GEO"].str.contains("TIME|Dataset|Last updated|Unit|Source", na=False)
        ]
        df = df[~df["GEO"].str.match(r"^\d+$", na=False)]

        # Extract every second column
        relevant_columns = ["GEO"] + [
            col for idx, col in enumerate(df.columns[1:]) if idx % 2 == 0
        ]
        df = df[relevant_columns]

        # Setting the index years
        df.columns = ["GEO"] + [
            str(year) for year in range(2013, 2013 + len(relevant_columns) - 1)
        ]

        # convert to float
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # drop fully empty rows
        df = df.dropna(how="all", subset=df.columns[1:])

        # transform to long fromat
        df = pd.melt(df, id_vars=["GEO"], var_name="Year", value_name="Value")

        # drop empty rows
        df = df.dropna(subset=["Value"])

        return df
    except Exception as e:
        print(f"Error with sheet {sheet_name}: {e}")
        return None


def load_crimes():
    # Inputfile and outputfolder
    input_file = "/kaggle/input/eurostat-datasets/Recorded-Crimes-until2022.xlsx"
    # Sheets to process
    sheets_to_process = [f"Sheet {i}" for i in range(1, 25, 2)]

    # Initializing empty dataframe
    merged_data = pd.DataFrame()

    # process every second sheet
    for sheet_name in sheets_to_process:
        try:
            # load sheets
            df = pd.read_excel(
                input_file, sheet_name=sheet_name, engine="openpyxl", header=None
            )

            # cleant the sheets
            processed_df = preprocess_sheet(sheet_name, df)

            if processed_df is not None:
                # include the cleaned sheet to the new outputfile
                merged_data = pd.concat([merged_data, processed_df], ignore_index=True)
        except Exception as e:
            print(f"Error loading Sheet {sheet_name}: {e}")

    # test if data is not empty
    if not merged_data.empty:
        # sum up the crimes
        aggregated_data = merged_data.groupby(["GEO", "Year"], as_index=False).sum()

        # save the sheet
        output_file = os.path.join("Recorded_Crimes_Total.xlsx")
        aggregated_data.to_excel(output_file, index=False)

        # print(f"Cleaned dataset saved to: {output_file}")
    else:
        print("No data found.")

##### Educational attainment

In [ ]:
input_file = "/kaggle/input/eurostat-datasets/Educational-Attainment.xlsx"

# relevant sheets
sheets_to_process = [
    "Sheet 19",
    "Sheet 20",
    "Sheet 21",
    "Sheet 22",
    "Sheet 23",
    "Sheet 24",
]

# Mapping
MONTHS_MAPPING = {
    "Jan": "01",
    "Feb": "02",
    "Mar": "03",
    "Apr": "04",
    "May": "05",
    "Jun": "06",
    "Jul": "07",
    "Aug": "08",
    "Sep": "09",
    "Oct": "10",
    "Okt": "10",
    "Nov": "11",
    "Dec": "12",
}

# Create a directory for the cleaned sheets
if not os.path.exists(output_folder):
    os.makedirs(output_folder)


def preprocess_sheet(sheet_name, df):
    """
    Preprocess an individual Excel sheet into a clean Pandas DataFrame.
    """
    try:
        # Drop all fully empty rows and columns
        df = df.dropna(how="all").reset_index(drop=True)

        # Find the row containing the header based on the presence of "TIME"
        header_idx = df[df.iloc[:, 0].astype(str).str.contains("TIME", na=False)].index
        if len(header_idx) == 0:
            raise ValueError(f"Header row not found in sheet: {sheet_name}")
        header_idx = header_idx[0]

        # Set the header row and rename columns
        df.columns = df.iloc[header_idx]
        df = df.iloc[header_idx + 1 :].reset_index(drop=True)

        # Rename the first column to "Country" for clarity
        df.rename(columns={df.columns[0]: "Country"}, inplace=True)

        # Melt the DataFrame into a long format (Country, Year, Value)
        df = df.melt(id_vars=["Country"], var_name="Year", value_name="Value")

        # Convert Year to numeric where possible
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

        # Handle string-based "Value" columns
        def clean_value(value):
            if isinstance(value, str):
                # Replace commas with dots for decimal points
                value = value.replace(",", ".")
                # Split the value if it contains a dot and a potential month
                if "." in value:
                    parts = value.split(".")
                    if len(parts) == 2 and parts[0] in MONTHS_MAPPING:
                        # If the first part matches a month, only keep the numeric part
                        return float(parts[1]) if parts[1].isdigit() else None
                # If it's a pure numeric string, convert to float
                if value.isdigit():
                    return float(value)
            return value

        df["Value"] = df["Value"].apply(clean_value)
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

        df = df.dropna(subset=["Country", "Year", "Value"]).reset_index(drop=True)

        # Final cleanup for Country column
        df["Country"] = df["Country"].str.strip()

        # Ensure proper formatting for the "Value" column
        df["Value"] = df["Value"].apply(
            lambda x: f"{x:.2f}" if isinstance(x, (int, float)) else x
        )

        # Set a unique index
        df.set_index(["Country", "Year"], inplace=True)

        return df
    except Exception as e:
        print(f"Error processing sheet {sheet_name}: {e}")
        return None


# Process each sheet
for sheet_name in sheets_to_process:
    print(f"Processing sheet: {sheet_name}")
    try:
        # Load the sheet into a Pandas DataFrame
        df = pd.read_excel(input_file, sheet_name=sheet_name, engine="openpyxl")

        # Read the value in cell C9 for naming the output file
        try:
            c9_value = pd.read_excel(
                input_file, sheet_name=sheet_name, engine="openpyxl", header=None
            ).iloc[8, 2]
            if pd.isna(c9_value):
                raise ValueError("Cell C9 is empty.")
            c9_value = str(c9_value).replace(" ", "_").strip()
        except Exception as e:
            print(f"Error reading cell C9 in sheet {sheet_name}: {e}")
            c9_value = sheet_name  # Fallback to sheet_name if C9 is invalid

        # Preprocess the sheet
        cleaned_df = preprocess_sheet(sheet_name, df)

        # Save the cleaned DataFrame to an Excel file
        if cleaned_df is not None:
            output_path = os.path.join(f"Educ-Attend-{c9_value}_cleaned_.xlsx")

            # Create an Excel workbook
            wb = Workbook()
            ws = wb.active
            ws.title = "Cleaned Data"

            # Format the "Value" column as text
            text_style = NamedStyle(name="text_style")
            text_style.number_format = "@"

            for col in ws.iter_cols(
                min_col=3, max_col=3
            ):  # Assuming "Value" is the third column
                for cell in col:
                    cell.style = text_style

            # Write data to the Excel sheet
            for row in dataframe_to_rows(
                cleaned_df.reset_index(), index=False, header=True
            ):
                ws.append(row)

            wb.save(output_path)
            print(f"Saved cleaned sheet: {output_path}")
        else:
            print(f"Skipped sheet due to errors: {sheet_name}")
    except Exception as e:
        print(f"Error reading sheet {sheet_name}: {e}")

##### GDP per capita

In [ ]:
file_path = "/kaggle/input/eurostat-datasets/GDP-percapita.xlsx"

raw_gdp = pd.read_excel(file_path, sheet_name="Sheet 1", header=None)

# define column names
columns = (
    ["GEO"]
    + [f"Year_{i}" for i in range(1995, 2024)]
    + [f"Unused_{i}" for i in range(1995, 2024)]
)
raw_gdp.columns = columns[: len(raw_gdp.columns)]

# drop irrelevant rows
clean_gdp = raw_gdp.dropna(subset=["GEO"])  # drop rows withount country
clean_gdp = clean_gdp[
    ~clean_gdp["GEO"].str.contains(
        "TIME|GEO|Dataset|Last updated|Unit|Source", na=False
    )
]
clean_gdp = clean_gdp[~clean_gdp["GEO"].str.match(r"^\d+$", na=False)]

# extract every second column
relevant_columns = ["GEO"] + [
    col for idx, col in enumerate(columns[1:]) if idx % 2 == 0
]
clean_gdp = clean_gdp[relevant_columns]

# rename columns correctly
clean_gdp.columns = ["GEO"] + [
    str(year) for year in range(1995, 1995 + len(relevant_columns) - 1)
]

# convert to float
for col in clean_gdp.columns[1:]:
    clean_gdp[col] = pd.to_numeric(clean_gdp[col], errors="coerce")

# drop empty rows
clean_gdp = clean_gdp.dropna(how="all", subset=clean_gdp.columns[1:])

# transform to long format
long_gdp = pd.melt(clean_gdp, id_vars=["GEO"], var_name="Year", value_name="Value")

# drop rows without values
long_gdp = long_gdp.dropna(subset=["Value"])

# sort the data
long_gdp = long_gdp.sort_values(by=["GEO", "Year"]).reset_index(drop=True)

# save the data

output_file_path = "GDP-percapita-cleaned_.xlsx"
long_gdp.to_excel(output_file_path, index=False)

print("Data successully saved!")

##### Unemployment

In [22]:
# load the data
file_path = "/kaggle/input/eurostat-datasets/Unemployment-3year-average.xlsx"
raw_data = pd.read_excel(file_path, sheet_name="Sheet 1", header=None)

# define column names
columns = ["GEO"] + [str(year) for year in range(2003, 2024)]
raw_data.columns = columns[: len(raw_data.columns)]

# clear metadata
clean_data = raw_data.dropna(subset=["GEO"])  # drop rows without country
clean_data = clean_data[
    ~clean_data["GEO"].str.contains("TIME|Dataset|Last updated|Unit|Source", na=False)
]

# drop metadatarows
clean_data = clean_data.iloc[1:, :]  # skip metadata

# replace : with NaN
clean_data.replace(":", pd.NA, inplace=True)

# change , to .
for col in clean_data.columns[1:]:
    clean_data[col] = clean_data[col].astype(str).str.replace(",", ".", regex=False)

    # convert to float
    clean_data[col] = pd.to_numeric(clean_data[col], errors="coerce")


# drop fully empty rows
clean_data = clean_data.dropna(how="all", subset=clean_data.columns[1:])

# transform to long format
long_data = pd.melt(clean_data, id_vars=["GEO"], var_name="Year", value_name="Value")

# drop rows without values
long_data = long_data.dropna(subset=["Value"])

# sort the data
long_data = long_data.sort_values(by=["GEO", "Year"]).reset_index(drop=True)

# save the file
output_file_path = "Unemployment-cleaned_.xlsx"
long_data.to_excel(output_file_path, index=False)

print("data successfully saved!")

data successfully saved!


In [ ]:
# Merging
abs
# List of input files
input_files = [
    "/kaggle/working/Educ-Attend-Less_than_primary,_primary_and_lower_secondary_education_(levels_0-2)_cleaned_.xlsx",
    "/kaggle/working/Educ-Attend-Tertiary_education_(levels_5-8)_cleaned_.xlsx",
    "/kaggle/working/Educ-Attend-Upper_secondary,_post-secondary_non-tertiary_and_tertiary_education_(levels_3-8)_cleaned_.xlsx",
    "/kaggle/working/Educ-Attend-Upper_secondary_and_post-secondary_non-tertiary_education_(levels_3_and_4)_-_general_cleaned_.xlsx",
    "/kaggle/working/Educ-Attend-Upper_secondary_and_post-secondary_non-tertiary_education_(levels_3_and_4)_-_vocational_cleaned_.xlsx",
    "/kaggle/working/Educ-Attend-Upper_secondary_and_post-secondary_non-tertiary_education_(levels_3_and_4)_cleaned_.xlsx",
    "/kaggle/working/GDP-percapita-cleaned_.xlsx",
    "/kaggle/working/Recorded_Crimes_Total.xlsx",
    "/kaggle/working/Unemployment-cleaned_.xlsx",
]

# Initialize an empty DataFrame
merged_data = None


# Function to generate meaningful column names based on the file name
def generate_column_name(file_name):
    if "Crimes" in file_name:
        return "Crime_Rate"
    elif "Less_than_primary" in file_name:
        return "Less_Primary_Education_Level"
    elif "Upper_secondary,_post-secondary_non-tertiary_and_tertiary" in file_name:
        return "Upper_Secondary_and_Tertiary_Education_Level"
    elif "general" in file_name:
        return "Upper_Secondary_General_Education_Level"
    elif "vocational" in file_name:
        return "Upper_Secondary_Vocational_Education_Level"
    elif "Upper_secondary_and_post-secondary_non-tertiary" in file_name:
        return "Upper_Secondary_Education_Level"
    elif "Tertiary_education" in file_name:
        return "Tertiary_Education_Level"
    elif "GDP" in file_name:
        return "GDP_Per_Capita"
    elif "Unemployment" in file_name:
        return "Unemployment_Rate"
    return "Value"


# Loop through all files
for file_path in input_files:
    if file_path.endswith(".xlsx"):
        # Load the data
        header_row = 1 if "Educ-Attend" in file_path else 0
        data = pd.read_excel(file_path, header=header_row)

        # Check and standardize column names
        if "Country" in data.columns:
            data.rename(columns={"Country": "GEO"}, inplace=True)
        if "year" in data.columns:
            data.rename(columns={"year": "Year"}, inplace=True)
        if "value" in data.columns:
            data.rename(columns={"value": "Value"}, inplace=True)

        # Ensure that required columns exist
        if not set(["GEO", "Year", "Value"]).issubset(data.columns):
            print(
                f"Warning: File {file_path} does not contain the required columns and will be skipped."
            )
            continue

        # Generate the column name based on the file name
        value_column_name = generate_column_name(file_path)

        # Rename the 'Value' column
        data = data.rename(columns={"Value": value_column_name})

        # Merge data based on GEO and Year
        if merged_data is None:
            merged_data = data
        else:
            merged_data = pd.merge(
                merged_data,
                data,
                on=["GEO", "Year"],
                how="outer",
                suffixes=(None, f"_{value_column_name}"),
            )

# Save the merged dataset
output_file = "Eurostat_Merged_Dataset.xlsx"
merged_data.to_excel(output_file, index=False)

print(f"The merged dataset has been successfully saved: {output_file}")


# File paths for input datasets
csv_file_path = "\kaggle\working\_q1q45.csv"
excel_file_path = "\kaggle\working\Eurostat_Merged_Dataset.xlsx"
output_file_path = "_MergedDatasetQ1andQ4.csv"

# Load the CSV file
csv_data = pd.read_csv(csv_file_path)
print("Loaded CSV data:")
print(csv_data.head())

# Load the Excel file
excel_data = pd.read_excel(excel_file_path, sheet_name=0)
print("Loaded Excel data:")
print(excel_data.head())

# Standardize column names for merging
csv_data.rename(columns={"Country": "GEO"}, inplace=True)

# Perform the merge on GEO and Year columns
merged_data = pd.merge(csv_data, excel_data, on=["GEO", "Year"], how="outer")
print("Merged data preview:")
print(merged_data.head())

# Save the merged data to a CSV file
merged_data.to_csv(output_file_path, index=False)
print(f"Merged dataset saved to {output_file_path}")

In [21]:
cpi = concat_cpis(load_all_cpi())

transparency = read_excel_corruption_risk("Transparency Index.xlsx", "T-Index scores")
transparency = cleaning_transparency(transparency)

# TODO make this cleaner
pi_19 = read_excel_corruption_risk("Public Integrity Index.xlsx", "2019")
pi_19 = cleaning_public_integrity(pi_19, 2019)

pi_21 = read_excel_corruption_risk("Public Integrity Index.xlsx", "2021")
pi_21 = cleaning_public_integrity(pi_21, 2021)

pi_23 = read_excel_corruption_risk("Public Integrity Index.xlsx", "2023")
pi_23 = cleaning_public_integrity(pi_23, 2023)

public_integrity = concat_public_integrity([pi_19, pi_21, pi_23])
public_integrity = missing_values_integrity(public_integrity)

eurobarometer = load_eurobarometer()

crimes = load_crimes()

GEO       object
Year      object
Value    float64
dtype: object


In [13]:
eurostat

,GEO,Year,Value
0,Albania,2013,7595.0
1,Albania,2014,10559.0
2,Albania,2015,10030.0
3,Albania,2016,12057.0
4,Albania,2017,10481.0
...,...,...,...
388,Türkiye,2018,384021.0
389,Türkiye,2019,333069.0
390,Türkiye,2020,307087.0
391,Türkiye,2021,320912.0


## 4. Explore the Data

### Exploratory Data Analysis (EDA)
- **Initial observations and summary statistics:**  
  Summary statistics revealed varying levels of corruption across countries, with clear discrepancies between perceived and actual corruption metrics. Composite indices such as the tolerance index (`QB4T`) provided aggregated insights.  
- **Identifying patterns and anomalies:**  
  - Heatmaps and clustering revealed regional corruption patterns.  
  - Scatterplots and bar plots highlighted sector-specific and country-level anomalies in both perceived and actual corruption.
  - CPI trends showed variations in corruption perceptions over time.

In [ ]:
def histograms(df: pd.DataFrame):
    """
    Histograms of all relevant metrics
    """
    for col in range(8, df.shape[1]):
        column_name = df.columns[col]
        print(column_name)
        plt.figure(figsize=(10, 6))
        plt.hist(df.iloc[:, col].dropna(), edgecolor="k", alpha=0.7)
        plt.title(f"{column_name} score distribution")
        plt.xlabel(column_name)
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.show()


def stacked_bar_chart(df: pd.DataFrame):
    """
    Stacked bar chart of non-null counts by region
    """
    region_col = df.columns[2]
    numeric_cols = df.columns[10:]

    # Count non-null values by region
    non_null_counts = df.groupby(region_col)[numeric_cols].apply(
        lambda x: x.notnull().sum()
    )

    # Plot bar chart
    non_null_counts.T.plot(kind="barh", stacked=True, figsize=(12, 8), colormap="tab10")
    plt.title("Non-null counts by region and metric")
    plt.ylabel("Metrics")
    plt.xlabel("Non-null count")
    plt.legend(title="Region", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


def number_of_sources_per_region(df: pd.DataFrame):
    """
    Bar chart of the number of sources per region
    """
    region_col = df.columns[2]
    numeric_cols = df.columns[10:]

    # Count non-null sources per row
    df["non_null_count"] = df[numeric_cols].notnull().sum(axis=1)

    # Sum counts per region
    sources_per_region = df.groupby(region_col)["non_null_count"].sum()

    # Sort the sources per region in descending order
    sources_per_region = sources_per_region.sort_values(ascending=False)

    # Plot bar chart
    sources_per_region.plot(kind="barh", figsize=(12, 8))
    plt.title("Sum of sources per region")
    plt.xlabel("Region")
    plt.ylabel("Available sources")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def source_fullness(df: pd.DataFrame):
    """
    Horizontal bar chart showing the source fullness per region
    """
    region_col = df.columns[2]
    numeric_cols = df.columns[9:]

    # Calculate source fullness
    df["source_fullness"] = df[numeric_cols].notnull().sum(axis=1) / len(numeric_cols)

    # Aggregate by region
    source_fullness_by_region = df.groupby(region_col)["source_fullness"].mean()

    # Plot horizontal bar chart
    plt.figure(figsize=(12, 8))
    source_fullness_by_region.sort_values().plot(kind="barh")
    plt.title("Source Fullness per Region")
    plt.xlabel("Source Fullness (Average)")
    plt.ylabel("Region")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def corr_map(df: pd.DataFrame):
    """
    Correlation matrix of CPI sources
    """
    numeric_cols = df.columns[10:]

    # Calculate correlation matrix
    corr = df[numeric_cols].corr()

    # Plot heatmap with symmetric colormap centered at 0
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f", center=0)
    plt.title("Correlation Matrix of CPI Sources")
    plt.tight_layout()
    plt.show()


def table_fullness(df: pd.DataFrame):
    """
    Heatmap showing the source fullness per year and region
    """
    region_col = df.columns[2]
    year_col = df.columns[3]
    numeric_cols = df.columns[10:]

    # Calculate source fullness
    df["source_fullness"] = df[numeric_cols].notnull().sum(axis=1) / len(numeric_cols)

    # Pivot table to get source fullness per year and region
    fullness_table = df.pivot_table(
        values="source_fullness", index=region_col, columns=year_col, aggfunc="mean"
    )

    # Plot heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        fullness_table,
        annot=True,
        cmap="viridis",
        vmin=3 / 13,
        # vmax=1,
        cbar_kws={"label": "Source Fullness"},
    )
    plt.title("Source Fullness per Year and Region")
    plt.xlabel("Year")
    plt.ylabel("Region")
    plt.tight_layout()
    plt.show()


def de_facto_de_jure_stacked(df: pd.DataFrame, year: int):
    # Filter data for the specified year
    df_year = df[df["Year"] == year]

    # Melt the DataFrame to long format for seaborn
    df_melted = df_year.melt(
        id_vars=["Country", "Region"],
        value_vars=["df_fulfillment", "dj_fulfillment"],
        var_name="Score Type",
        value_name="Score",
    )

    # Sort by df_totalscore in descending order
    df_melted = df_melted.sort_values(by="Score", ascending=False)

    # Create a FacetGrid with one subplot per region
    g = sns.FacetGrid(
        df_melted,
        col="Region",
        col_wrap=3,
        sharex=True,
        sharey=False,
        height=4,
        aspect=1.5,
    )
    g.map_dataframe(
        sns.barplot,
        y="Country",
        x="Score",
        hue="Score Type",
        palette="viridis",
        errorbar=None,
        orient="h",
    )

    # Adjust the plot
    g.add_legend(title="Score Type")
    g.set_titles("{col_name}")
    g.set_xticklabels(rotation=0)
    g.set_axis_labels("Score", "Country")
    plt.subplots_adjust(top=0.9)
    g.figure.suptitle("De facto and De jure scores by Region", fontsize=16)
    plt.show()


def de_facto_de_jure_colored(df: pd.DataFrame, year: int):
    # Filter data for the specified year
    df_year = df[df["Year"] == year]

    # Sort by df_fulfillment in descending order
    df_year = df_year.sort_values(by="df_fulfillment", ascending=False)

    # Melt the DataFrame to long format for seaborn
    df_melted = df_year.melt(
        id_vars=["Country", "Region"],
        value_vars=["df_fulfillment", "dj_fulfillment"],
        var_name="Score Type",
        value_name="Score",
    )

    # Create a bar plot
    plt.figure(figsize=(10, 24))
    barplot = sns.barplot(
        data=df_melted,
        x="Score",
        y="Country",
        hue="Score Type",
        palette="viridis",
        errorbar=None,
    )

    # Color the country labels for EU countries
    for i, country in enumerate(df_year["Country"]):
        if df_year.iloc[i]["Region"] == "WE/EU":
            barplot.get_yticklabels()[i].set_color("red")

    # Adjust the plot
    plt.title("De facto and De jure scores by country")
    plt.xlabel("Score")
    plt.ylabel("Country")
    plt.legend(title="Score Type", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO, More vizs here, integrate with above
QUESTION_DICT = {
    "QB1": "Visited Public Healthcare Practitioner (Last 12 Months)",
    "QB2": "Extra Payment or Gift to Public Healthcare Practitioner",
    "QB3": "Description of Extra Payment Incident",
    "QB4.1": "Acceptability of Giving Money (Public Services)",
    "QB4.2": "Acceptability of Giving a Gift (Public Services)",
    "QB4.3": "Acceptability of Doing a Favor (Public Services)",
    "QB4T": "Tolerance Index to Corruption",
    "QB5": "Perceived Widespread Corruption",
    "QB6": "Corruption Trend in the Last 3 Years",
    "QB7": "Corruption Widespread in Sectors",
    "QB8": "Knows Someone Who Takes Bribes",
    "QB10": "Knows Where to Report Corruption",
    "QB12": "Experienced or Witnessed Corruption",
    "QB_avg": "Average of All QB Questions",
    "CPI_avg": "Average Corruption Perception Index (2020-2023)",
    "CPI_percent_change": "CPI Percent Change (2020-2023)",
    "Latitude": "Latitude (Country)",
    "Longitude": "Longitude (Country)",
}


# Load preprocessed data
def load_preprocessed_data(file_path):
    return pd.read_csv(file_path)


# Descriptive statistics
def descriptive_statistics(df):
    stats = df.describe().T
    stats["missing_values"] = df.isnull().sum()
    print("Descriptive Statistics:")
    print(stats)
    stats.to_csv("descriptive_statistics.csv")
    print("Saved descriptive statistics to descriptive_statistics.csv")


# Generate geographical analysis with country labels
def geographical_analysis(df):
    plt.figure(figsize=(15, 10))

    # Create scatter plot with country labels
    for _, row in df.iterrows():
        plt.scatter(
            row["Longitude"], row["Latitude"], alpha=0.7, s=50, label=row["Country"]
        )
        plt.text(row["Longitude"] + 0.2, row["Latitude"], row["Country"], fontsize=9)

    # Adding labels and grid
    plt.title("Geographical Distribution of Data in Europe")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.grid(True)

    # Approximate bounds for Europe
    plt.xlim(-25, 45)
    plt.ylim(35, 70)

    # Save and show plot
    plt.savefig("geographical_distribution_europe_with_labels.png")
    plt.show()
    print(
        "Saved geographical distribution map to geographical_distribution_europe_with_labels.png"
    )


# Heatmap based on latitude and longitude
def create_geographical_heatmap(df, value_column):
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(
        df["Longitude"],
        df["Latitude"],
        c=df[value_column],
        cmap="viridis",
        s=100,
        alpha=0.75,
    )
    for _, row in df.iterrows():
        plt.text(row["Longitude"] + 0.2, row["Latitude"], row["Country"], fontsize=8)
    plt.colorbar(scatter, label=QUESTION_DICT.get(value_column, value_column))
    plt.title(
        f"Heatmap of {QUESTION_DICT.get(value_column, value_column)} by Geographical Coordinates"
    )
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.grid(True)
    plt.savefig(f"heatmap_{value_column}.png")
    plt.show()
    print(f"Saved heatmap of {value_column} to heatmap_{value_column}.png")


def histograms_per_country(df, question_columns):
    for question in question_columns:
        plt.figure(figsize=(12, 8))

        # Aggregate data by country for the question
        grouped_data = df.groupby("Country")[question].mean().sort_values()

        # Create bar plot
        grouped_data.plot(kind="bar", color="skyblue", edgecolor="black")
        plt.title(f"Bar Plot of {question} by Country")
        plt.xlabel("Country")
        plt.ylabel(question)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(f"barplot_{question}_by_country.png")
        plt.show()
        print(
            f"Saved bar plot for {question} by country to barplot_{question}_by_country.png"
        )


# Pairplot for selected variables
# Pairplot for selected variables with better axis titles
def create_pairplot(df, columns):
    # Map column codes to descriptive titles with line breaks
    column_titles = [QUESTION_DICT.get(col, col).replace(" ", "\n") for col in columns]
    df_renamed = df[columns].rename(columns=dict(zip(columns, column_titles)))

    # Create pairplot
    g = sns.pairplot(
        df_renamed, diag_kind="kde", corner=True, height=3
    )  # Adjust height for larger plots

    # Adjust plot settings for better readability
    g.fig.set_size_inches(12, 12)  # Increase the figure size
    g.fig.subplots_adjust(top=0.95)  # Adjust space for title
    g.fig.suptitle(
        "Pairplot of Selected Questions with Descriptive Titles", fontsize=16
    )

    # Save and display the pairplot
    plt.savefig("pairplot_with_readable_titles.png", dpi=300)  # High-resolution save
    plt.show()
    print("Saved pairplot with readable titles to pairplot_with_readable_titles.png")


# CPI bar plots
def cpi_bar_plot(df, cpi_columns):
    melted_df = df.melt(
        id_vars=["Country"], value_vars=cpi_columns, var_name="Year", value_name="CPI"
    )
    plt.figure(figsize=(15, 8))
    sns.barplot(data=melted_df, x="Country", y="CPI", hue="Year", palette="viridis")
    plt.title("CPI by Country and Year")
    plt.xlabel("Country")
    plt.ylabel("CPI")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("cpi_bar_plot.png")
    plt.show()
    print("Saved CPI bar plot to cpi_bar_plot.png")


# Histogram of Composite Scores
def histogram_composite_scores(df, columns):
    for col in columns:
        plt.figure(figsize=(10, 6))
        sns.histplot(df[col], kde=True, bins=20)
        plt.title(f"Distribution of {QUESTION_DICT.get(col, col)}")
        plt.xlabel(QUESTION_DICT.get(col, col))
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.savefig(f"histogram_{col}.png")
        plt.show()
        print(f"Saved histogram for {col} to histogram_{col}.png")


# Histogram per country for CPI by year
def cpi_histograms_by_country(df, cpi_columns):
    for year_column in cpi_columns:
        plt.figure(figsize=(15, 8))
        sns.histplot(
            data=df,
            x=year_column,
            hue="Country",
            multiple="stack",
            palette="viridis",
            kde=False,
            bins=20,
        )
        plt.title(f"Histogram of {year_column} by Country")
        plt.xlabel(QUESTION_DICT.get(year_column, year_column))
        plt.ylabel("Frequency")
        plt.legend(title="Country", loc="upper right", bbox_to_anchor=(1.15, 1))
        plt.tight_layout()
        plt.savefig(f"cpi_histogram_{year_column}.png")
        plt.show()
        print(f"Saved histogram for {year_column} to cpi_histogram_{year_column}.png")


def bar_plot_by_country(df, question_columns):
    for question in question_columns:
        plt.figure(figsize=(12, 8))

        grouped_data = df.groupby("Country")[question].mean().sort_values()

        grouped_data.plot(kind="bar", color="skyblue", edgecolor="black")
        plt.title(f"Bar Plot of {QUESTION_DICT.get(question, question)} by Country")
        plt.xlabel("Country")
        plt.ylabel(QUESTION_DICT.get(question, question))
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(f"barplot_{question}_by_country.png")
        plt.show()
        print(
            f"Saved bar plot for {question} by country to barplot_{question}_by_country.png"
        )


def main():
    # Define the path to the file in the working directory
    file_path = "/kaggle/working/cleaned_corruption_data_v2.csv"

    # Load the data
    CORRUPTION_DF = load_preprocessed_data(file_path)
    print("Data loaded successfully!")
    print(CORRUPTION_DF.head())

    descriptive_statistics(CORRUPTION_DF)

    # Numerical columns for analysis
    numerical_columns = [
        col
        for col in CORRUPTION_DF.columns
        if CORRUPTION_DF[col].dtype in ["float64", "int64"]
    ]

    # Geographical Analysis
    geographical_analysis(CORRUPTION_DF)

    # Create Heatmaps for Specific Questions
    for col in ["QB1", "QB2", "QB10"]:
        create_geographical_heatmap(CORRUPTION_DF, col)

    # Pairplot for Selected Variables
    pairplot_columns = ["QB1", "QB2", "QB10", "CPI 2020", "CPI 2023"]
    create_pairplot(CORRUPTION_DF, pairplot_columns)

    # CPI Bar Plot
    cpi_columns = ["CPI 2020", "CPI 2021", "CPI 2022", "CPI 2023"]
    cpi_bar_plot(CORRUPTION_DF, cpi_columns)

    # Line Plot for CPI Trends
    cpi_columns = ["CPI 2020", "CPI 2021", "CPI 2022", "CPI 2023"]
    line_plot_cpi(CORRUPTION_DF, cpi_columns)

    # Bar Plots for QB1, QB2, and QB3 by Country
    question_columns = ["QB1", "QB2"]
    bar_plot_by_country(CORRUPTION_DF, question_columns)

    # Histogram of Composite Scores
    composite_columns = ["QB_avg", "CPI_avg"]
    histogram_composite_scores(CORRUPTION_DF, composite_columns)


In [ ]:
main()

## 4. Model the Data

### Building Predictive Models
- Overview of models used
- Metrics for evaluation

### Identifying Key Predictors
- Analysis of features contributing to corruption levels

### Interpretation of Results
- Insights from the models
- Discussion of their validity and reliability

#### CPI

In [ ]:
def basic_prep(df: pd.DataFrame) -> pd.DataFrame:
    # Drop useless columns
    df = df.drop(
        columns=[
            "Region",
            "Country",
            "Rank",
            "Number of sources",
            "Standard error",
            "ISO3",
            "Upper CI",
            "Lower CI",
        ]
    )
    return df


def data_prep(
    df: pd.DataFrame,
    without_year: bool = True,
    row_nan_drop: bool = True,
    europe: bool = True,
) -> pd.DataFrame:
    """
    Prepare the CPI dataset for modeling
    """
    # Consider only WE/EU region
    if europe:
        df = df[df["Region"] == "WE/EU"]
        # Drop columns with more than 100 NaNs
        df = df.drop(columns=df.columns[10:][df.iloc[:, 10:].isna().sum() >= 100])

    if without_year:
        df = df.drop(columns=["Year"])

    if row_nan_drop:
        df = df.dropna()

    return basic_prep(df)


def splitting(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataset into features and target
    """
    y = df["CPI score"]
    X = df.drop(columns=["CPI score"])

    return X, y


def train_and_evaluate_model(
    model,
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_test: pd.DataFrame,
):
    """
    Train the model using cross-validation and evaluate on the test set
    """
    # Perform cross-validation
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=5, scoring="neg_mean_squared_error"
    )
    print(f"{model_name} CV   MSE: {-cv_scores.mean()}")

    # Train the model on the full training set
    model.fit(X_train, y_train)

    # Evaluate the model on the test set
    y_pred = model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)
    print(f"{model_name} Test MSE: {test_mse}")
    print()

    return model


def cpi_models(cpi: pd.DataFrame):
    cpi_europe_nonan = data_prep(cpi)
    cpi_europe_nan = data_prep(cpi, row_nan_drop=False)

    cpi_all_nonan = data_prep(cpi, europe=False)
    cpi_all_nan = data_prep(cpi, row_nan_drop=False, europe=False)

    # Compare their shapes and NaN count
    print("Europe nonan shape:", cpi_europe_nonan.shape)
    print("Europe nan shape:", cpi_europe_nan.shape)
    print("All nonan shape:", cpi_all_nonan.shape)
    print("All nan shape:", cpi_all_nan.shape)
    print()
    print("Europe nonan NaNs:", cpi_europe_nonan.isna().sum().sum())
    print("Europe nan NaNs:", cpi_europe_nan.isna().sum().sum())
    print("All nonan NaNs:", cpi_all_nonan.isna().sum().sum())
    print("All nan NaNs:", cpi_all_nan.isna().sum().sum())
    print()

    # Split the datasets
    X_europe_nonan, y_europe_nonan = splitting(cpi_europe_nonan)
    X_europe_nan, y_europe_nan = splitting(cpi_europe_nan)
    X_all_nonan, y_all_nonan = splitting(cpi_all_nonan)  # This dataset is empty
    X_all_nan, y_all_nan = splitting(cpi_all_nan)

    # Split the datasets into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_europe_nonan, y_europe_nonan, test_size=0.2, random_state=SEED
    )

    # Train the linear models
    linear_model = train_and_evaluate_model(
        LinearRegression(), "Linear Regression", X_train, y_train, X_test, y_test
    )
    ridge_model = train_and_evaluate_model(
        Ridge(), "Ridge Regression", X_train, y_train, X_test, y_test
    )
    lasso_model = train_and_evaluate_model(
        Lasso(), "Lasso Regression", X_train, y_train, X_test, y_test
    )

    # Train the gradient boosting model
    gb_model = train_and_evaluate_model(
        HistGradientBoostingRegressor(random_state=SEED, max_iter=300),
        "Gradient Boosting",
        X_train,
        y_train,
        X_test,
        y_test,
    )

    # Train the gradient boosting model with the whole dataset
    X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
        X_all_nan, y_all_nan, test_size=0.2, random_state=SEED
    )

    gb_model_all = train_and_evaluate_model(
        HistGradientBoostingRegressor(random_state=SEED, max_iter=300),
        "Gradient Boosting with all countries",
        X_train_all,
        y_train_all,
        X_test_all,
        y_test_all,
    )

#### Eurostat

In [ ]:
dataset_path = "/kaggle/working/_MergedDatasetQ1andQ4.csv"
dataraw = pd.read_csv(dataset_path)
selected_features = [
    "Unemployment_Rate",
    "Upper_Secondary_Vocational_Education_Level",
    "Upper_Secondary_General_Education_Level",
    "Upper_Secondary_Education_Level",
    "Less_Primary_Education_Level",
    "Tertiary_Education_Level",
    "Crime_Rate",
    "GDP_Per_Capita",
    "mean years of schooling",
    "urban population",
    "population",
    "life expectancy",
]

target_column = "df_fulfillment"
# aggregated features on GEO as more records of features than df_fulfillment
aggregated_features = dataraw.groupby("GEO")[selected_features].mean().reset_index()

# merge with the df_fulfillment
data = pd.merge(
    aggregated_features, dataraw[["GEO", "df_fulfillment"]].dropna(), on="GEO"
)
# Define features (X) and target (y)
X = data[selected_features]
y = data[target_column]

# Initialize manual grid search for XGBoost
kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_score = -float("inf")
best_params = None
# Parameter grid for XGBoost
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0],
}

# Perform manual cross-validation
for n_estimators in param_grid["n_estimators"]:
    for learning_rate in param_grid["learning_rate"]:
        for max_depth in param_grid["max_depth"]:
            for subsample in param_grid["subsample"]:
                model = XGBRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=subsample,
                    random_state=42,
                    use_label_encoder=False,
                )
                cv_scores = []
                for train_idx, val_idx in kf.split(X):
                    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
                    model.fit(X_train, y_train)
                    predictions = model.predict(X_val)
                    cv_scores.append(r2_score(y_val, predictions))
                mean_score = np.mean(cv_scores)
                if mean_score > best_score:
                    best_score = mean_score
                    best_params = {
                        "n_estimators": n_estimators,
                        "learning_rate": learning_rate,
                        "max_depth": max_depth,
                        "subsample": subsample,
                    }
# Train the final model with the best parameters
final_model = XGBRegressor(
    **best_params,
    random_state=42,
)
final_model.fit(X, y)

# Evaluate the final model
predictions = final_model.predict(X)
mse = mean_squared_error(y, predictions)
r2 = r2_score(y, predictions)

# Output results
print("Best Parameters:", best_params)
print("Mean Squared Error:", mse)
print("R² Score:", r2)

#### Perceived vs actual

In [ ]:
# TODO add here modeling for this part (see above)

In [ ]:
# TODO add here the "mains" for running all modeling

cpi_models(cpi)


## 5. Communicate and Visualize Results

### Key Findings and Insights
- Summary of answers to the research questions

### Visualizations
- Graphs, charts, and tables illustrating the results

### Biases and Limitations
- Discussion of potential biases in data and analysis

---

#### Corruption Risk

## 6. Reflection and Learning

### Challenges Faced
- Difficulties encountered during analysis

### Insights and Biases
- Key takeaways from the analysis
- Potential biases identified

### Techniques and Tools
- Summary of data science tools and methods learned

### Division of Work
- Contribution of group members

---